In [1]:
!pip install json-repair

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.6/50.6 kB 2.7 MB/s eta 0:00:00


In [2]:
import json
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import re
from typing import List, Literal
from pydantic import BaseModel, Field
from json_repair import repair_json

In [3]:
data=[]
with open("/kaggle/input/datasets/koushikikundu/hr-policy/hr_policy_qa.jsonl", "r") as files:
    for file in files:
        data.append(json.loads(file))

In [4]:
len(data)

71

In [5]:
class Message(BaseModel):
    role: Literal["user", "assistant"]
    content: str

class Conversation(BaseModel):
    messages: List[Message]

class SyntheticDataset(BaseModel):
    conversations: List[Conversation] = Field(
        min_length=5,
        max_length=5
    )

In [6]:
# Prompt Generator Function
def prompt_gen(question: str, answer: str) -> str:
    schema = json.dumps(SyntheticDataset.model_json_schema(), indent=2)
    return f"""You are creating synthetic training data.

Original question:
{question}

Original answer:
{answer}

Generate exactly 5 new question-answer pairs based on this.

Rules:
1. The answer must NEVER introduce new facts.
2. The answer must contain ONLY information present in the original answer.
3. Rewrite or paraphrase the question.
4. Do NOT invent scenarios that require information not present in the answer.

You MUST respond with strictly valid JSON matching this schema:
{schema}

Example Output Format:
{{
  "conversations": [
    {{
      "messages": [
        {{
          "role": "user",
          "content": "<question>"
        }},
        {{
          "role": "assistant",
          "content": "<answer>"
        }}
      ]
    }}
  ]
}}"""


In [7]:
# 3. Load Model and Tokenizer
model_name = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.bfloat16, device_map="auto"
)

all_qa_pairs = []
for chunk in data:
    for qa_pair in chunk:
        all_qa_pairs.append(qa_pair)

BATCH_SIZE = 8  # Adjust based on VRAM (e.g., 4, 8, or 16)
generated_results = []

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [8]:
for i in range(0, len(all_qa_pairs), BATCH_SIZE):
    batch_qa = all_qa_pairs[i : i + BATCH_SIZE]

    # Build chat messages for each item in the batch
    batch_formatted_prompts = []
    for qa in batch_qa:
        user_prompt = prompt_gen(qa["question"], qa["answer"])
        messages = [
            {
                "role": "system",
                "content": "You are a synthetic data generator. You output strictly valid JSON matching the requested schema.",
            },
            {"role": "user", "content": user_prompt},
        ]
        formatted = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        batch_formatted_prompts.append(formatted)

    # Tokenize the entire batch with padding
    model_inputs = tokenizer(
        batch_formatted_prompts, return_tensors="pt", padding=True
    ).to(model.device)

    # Generate tokens for all batch items in parallel
    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=1024,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )

    # Decode and parse outputs
    input_length = model_inputs.input_ids.shape[1]
    for idx, output_ids in enumerate(generated_ids):
        # Extract only generated tokens
        response_text = tokenizer.decode(
            output_ids[input_length:], skip_special_tokens=True
        ).strip()

        # Clean markdown wrappers if present
        response_text = re.sub(
            r"^```(?:json)?\s*|\s*```$",
            "",
            response_text,
            flags=re.DOTALL,
        ).strip()
        # Parse with Pydantic
        try:
            repaired = repair_json(response_text)
            dataset = SyntheticDataset.model_validate_json(repaired)
            generated_results.append(dataset.model_dump())
            print(
                f"[{i + idx + 1}/{len(all_qa_pairs)}] Parsed successfully!"
            )
        except Exception as e:
            print(f"[{i + idx + 1}/{len(all_qa_pairs)}] Validation Error:", e)

print(f"\nCompleted! Generated {len(generated_results)} datasets.")

[1/355] Parsed successfully!
[2/355] Parsed successfully!
[3/355] Parsed successfully!
[4/355] Parsed successfully!
[5/355] Parsed successfully!
[6/355] Parsed successfully!
[7/355] Parsed successfully!
[8/355] Parsed successfully!
[9/355] Parsed successfully!
[10/355] Parsed successfully!
[11/355] Parsed successfully!
[12/355] Parsed successfully!
[13/355] Parsed successfully!
[14/355] Parsed successfully!
[15/355] Parsed successfully!
[16/355] Parsed successfully!
[17/355] Parsed successfully!
[18/355] Parsed successfully!
[19/355] Parsed successfully!
[20/355] Parsed successfully!
[21/355] Parsed successfully!
[22/355] Parsed successfully!
[23/355] Parsed successfully!
[24/355] Parsed successfully!
[25/355] Parsed successfully!
[26/355] Parsed successfully!
[27/355] Parsed successfully!
[28/355] Parsed successfully!
[29/355] Parsed successfully!
[30/355] Parsed successfully!
[31/355] Parsed successfully!
[32/355] Parsed successfully!
[33/355] Parsed successfully!
[34/355] Parsed suc

In [9]:
len(generated_results)

354

In [10]:
with open("hr_policy_synthetic_qa.jsonl", "w", encoding="utf-8") as f:
            f.write(json.dumps(generated_results, ensure_ascii=False) + "\n")
            print("Inserted")

Inserted
